# V17 — Threshold τ Sweep (tab:threshold)
**Professor D**  
Goal: Find the empirical collapse threshold — at what τ does BMIA-A remain stable?  
Sweeps τ ∈ {0.05, 0.10, 0.15, 0.20, 0.25} × LR ∈ {0.005, 0.01, 0.02, 0.05}  
Output → `V17_THRESHOLD.json`

**Kaggle**: Add Data → `hendrycks/cifar100c` | GPU T4 x2 | Internet ON | ~30 min

In [ ]:
import os,json,copy,gc,time
import numpy as np
import torch,torch.nn as nn,torch.optim as optim
import torchvision,torchvision.transforms as T
from torch.utils.data import DataLoader

torch.manual_seed(42); np.random.seed(42)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS=torch.cuda.device_count()
print(f'Device: {device}  GPUs: {N_GPUS}')

C100C_ROOT=None
for root,dirs,files in os.walk('/kaggle/input/'):
    if 'gaussian_noise.npy' in files and 'labels.npy' in files:
        C100C_ROOT=root; break
assert C100C_ROOT,'Add dataset: hendrycks/cifar100c'

NUM_CLASSES=100; TTA_BATCH=64; EVAL_BATCH=512
SEVERITY=5; SEV_OFFSET=(SEVERITY-1)*10000
CORRUPTIONS=['gaussian_noise','defocus_blur','snow','contrast','jpeg_compression']
TAU_LIST=[0.05,0.10,0.15,0.20,0.25]
LR_LIST =[0.005,0.01,0.02,0.05]
C_MEAN=torch.tensor([0.5071,0.4867,0.4408]).view(1,3,1,1)
C_STD =torch.tensor([0.2675,0.2565,0.2761]).view(1,3,1,1)

def load_c100c(c):
    data=np.load(f'{C100C_ROOT}/{c}.npy')[SEV_OFFSET:SEV_OFFSET+10000]
    lbl =np.load(f'{C100C_ROOT}/labels.npy')
    X=(torch.tensor(data).permute(0,3,1,2).float()/255.0-C_MEAN)/C_STD
    return X.to(device),torch.tensor(lbl,dtype=torch.long).to(device)

print(f'TAU_LIST={TAU_LIST}  LR_LIST={LR_LIST}')
print(f'Total runs: {len(TAU_LIST)}×{len(LR_LIST)}×{len(CORRUPTIONS)} = {len(TAU_LIST)*len(LR_LIST)*len(CORRUPTIONS)}')

In [ ]:
# ── Train ResNet-18 (3×3 conv1, 30 ep) ───────────────────────────
ds=torchvision.datasets.CIFAR100('/tmp',train=True,download=True,transform=T.Compose([
    T.RandomCrop(32,4),T.RandomHorizontalFlip(),T.ToTensor(),
    T.Normalize([0.5071,0.4867,0.4408],[0.2675,0.2565,0.2761])]))
dl=DataLoader(ds,batch_size=128,shuffle=True,num_workers=4,pin_memory=True)

model=torchvision.models.resnet18(weights=None)
model.conv1=nn.Conv2d(3,64,3,stride=1,padding=1,bias=False)
model.maxpool=nn.Identity()
model.fc=nn.Linear(512,NUM_CLASSES)
model=model.to(device)
dp=nn.DataParallel(model) if N_GPUS>1 else model
opt_tr=optim.SGD(model.parameters(),lr=0.1,momentum=0.9,weight_decay=5e-4)
sch=optim.lr_scheduler.CosineAnnealingLR(opt_tr,T_max=30)
crit=nn.CrossEntropyLoss()
dp.train(); t0=time.time()
for ep in range(30):
    for x,y in dl:
        x,y=x.to(device),y.to(device)
        opt_tr.zero_grad(); crit(dp(x),y).backward(); opt_tr.step()
    sch.step()
    if (ep+1)%10==0: print(f'  ep={ep+1}/30  {(time.time()-t0)/60:.1f}min')
model.eval(); SOURCE_STATE=copy.deepcopy(model.state_dict())
del dp,dl,ds; gc.collect(); torch.cuda.empty_cache()
print('Training done.')

In [ ]:
# ── Helpers ──────────────────────────────────────────────────────
def freeze_bn(mdl):
    ids={id(p) for m in mdl.modules() if isinstance(m,nn.BatchNorm2d) for p in m.parameters()}
    for p in mdl.parameters(): p.requires_grad_(id(p) in ids)

def get_acc(mdl,X,y):
    mdl.eval(); preds=[]
    with torch.no_grad():
        for i in range(0,X.size(0),EVAL_BATCH): preds.append(mdl(X[i:i+EVAL_BATCH]).argmax(1))
    p=torch.cat(preds)
    nc=(torch.bincount(p,minlength=NUM_CLASSES)>0).sum().item()
    dom=torch.bincount(p,minlength=NUM_CLASSES).float().max().item()/p.size(0)
    acc=(p==y[:len(p)]).float().mean().item()
    return round(acc,4),round(dom,4),nc

def run_bmia_adaptive(X,y,lr,tau):
    """BMIA-Adaptive with configurable collapse threshold τ."""
    model.load_state_dict(copy.deepcopy(SOURCE_STATE))
    model.train(); freeze_bn(model)
    params=[p for p in model.parameters() if p.requires_grad]
    opt=optim.SGD(params,lr=lr,momentum=0.9)
    init={pn:p.data.clone() for pn,p in model.named_parameters() if p.requires_grad}
    lam=0.5
    for i in range(0,X.size(0),TTA_BATCH):
        xb=X[i:i+TTA_BATCH]
        if xb.size(0)<4: continue
        opt.zero_grad()
        pr=torch.softmax(model(xb),1); ent=-(pr*torch.log(pr+1e-8)).sum(1)
        mg=pr.mean(0); hy=-(mg*torch.log(mg+1e-8)).sum()
        anc=sum(((p-init[pn])**2).sum() for pn,p in model.named_parameters() if pn in init)
        (ent.mean()-hy+lam*anc).backward(); opt.step()
        with torch.no_grad():
            dom=(torch.bincount(pr.argmax(1),minlength=NUM_CLASSES).float().max()/xb.size(0)).item()
        lam=min(10.,lam*1.1) if dom>tau else max(0.01,lam*0.95)
    return get_acc(model,X,y)

print('Helpers ready.')

In [ ]:
# ── Run τ sweep ──────────────────────────────────────────────────
results=[]; t0=time.time()
for corr in CORRUPTIONS:
    X,y=load_c100c(corr)
    print(f'\n=== {corr} ===')
    for lr in LR_LIST:
        for tau in TAU_LIST:
            acc,dom,nc=run_bmia_adaptive(X,y,lr,tau)
            col=(dom>0.15 and nc<50) or nc<20
            results.append({'corruption':corr,'lr':lr,'tau':tau,
                            'acc':acc,'dom_pct':dom,'n_classes':nc,'collapsed':col})
            print(f'  lr={lr}  τ={tau}  acc={acc*100:.1f}%  dom={dom*100:.1f}%  '
                  f'cls={nc}  {"COLLAPSED" if col else "ok"}')
            gc.collect(); torch.cuda.empty_cache()
    del X,y; gc.collect(); torch.cuda.empty_cache()
print(f'Done {(time.time()-t0)/60:.1f}min')

# ── Summary table ─────────────────────────────────────────────────
print('\n=== AVG ACCURACY  (rows=τ, cols=lr) ===')
print(f'{"τ":<6}',end='')
for lr in LR_LIST: print(f'  lr={lr}',end='')
print()
for tau in TAU_LIST:
    print(f'{tau:<6}',end='')
    for lr in LR_LIST:
        vals=[r['acc'] for r in results if r['tau']==tau and r['lr']==lr]
        print(f'  {np.mean(vals)*100:5.1f}%',end='')
    print()

print('\n=== COLLAPSE COUNT (rows=τ, cols=lr) ===')
print(f'{"τ":<6}',end='')
for lr in LR_LIST: print(f'  lr={lr}',end='')
print()
for tau in TAU_LIST:
    print(f'{tau:<6}',end='')
    for lr in LR_LIST:
        nc=sum(1 for r in results if r['tau']==tau and r['lr']==lr and r['collapsed'])
        n =sum(1 for r in results if r['tau']==tau and r['lr']==lr)
        print(f'  {nc}/{n}',end='')
    print()

with open('V17_THRESHOLD.json','w') as f:
    json.dump({'experiment':'V17_THRESHOLD','tau_list':TAU_LIST,'lr_list':LR_LIST,
               'severity':SEVERITY,'corruptions':CORRUPTIONS,'results':results},f,indent=2)
print('Saved → V17_THRESHOLD.json')
with open('V17_THRESHOLD.json') as f: print(f.read())